# OhDeer Haar and SVM Detection Baseline

CS566 Lecture 11 based classical localization, not age or antler classification.
This notebook is self-contained: it embeds the same source package as the local
project. No GitHub push or local C: drive is required. Use **Runtime → Run all**
in CPU mode to run the default synthetic software verification.

**The Snapshot Wisconsin v2 annotation fields have been inspected.** Synthetic
metrics are NOT deer results. Real mode requires the competition data,
checking the annotation format and completeness, and using a team-agreed split.
This notebook has been designed for Colab, but a Google-hosted session must still
be executed by the user; local notebook execution is not proof of cloud execution.


In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, os
if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10 or newer is required")
# Optional reproducible CPU thread limit; must be set before scientific imports.
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(variable, "1")
requirements = ["numpy>=1.24,<3", "scipy>=1.10,<2", "scikit-learn>=1.2,<2",
                "Pillow>=9.5,<13", "matplotlib>=3.7,<4", "joblib>=1.2,<2"]
# A clean local test environment already installed these. On Colab install or
# reconcile dependencies before importing the package. No GPU is required.
try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    in_colab = False
if in_colab:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *requirements])
else:
    missing = [name for name in ["numpy", "scipy", "sklearn", "PIL", "matplotlib", "joblib"]
               if importlib.util.find_spec(name) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])
print("Python", sys.version.split()[0])


In [ ]:
import base64, io, zipfile, hashlib, json
SOURCE_ARCHIVE = "..."
PROJECT = Path.cwd() / "ohdeer_colab_project"
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as archive:
    for member in archive.infolist():
        target = (PROJECT / member.filename).resolve()
        if not target.is_relative_to(PROJECT.resolve()):
            raise ValueError("Unsafe embedded path")
    archive.extractall(PROJECT)
hashes = json.loads((PROJECT / "SOURCE_SHA256.json").read_text())
assert all(hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == sha
           for name, sha in hashes.items())
sys.path.insert(0, str(PROJECT))
def run(*arguments):
    subprocess.run([sys.executable, "-m", "ohdeer", *map(str, arguments)],
                   cwd=PROJECT, check=True)
print("Verified embedded source files:", len(hashes))


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
               cwd=PROJECT, check=True)


## Choose demo or real data

Leave `MODE="demo"` for a self-contained end-to-end check. To use Kaggle,
change MODE to real, configure the following cells, then run again. Demo is
deliberately labeled synthetic in all reports and figures.


In [ ]:
MODE = "demo"  # Change to "real" ONLY after preparing actual data.
DATA_ROOT = Path("/content/ohdeer_data")
OUTPUT = PROJECT / "outputs" / ("demo" if MODE == "demo" else "real")
assert MODE in ("demo", "real")


## Optional real data download or extraction

For competition data, first log in to Kaggle and accept the competition rules.
The simplest option is to upload the downloaded ZIP to Colab using the Files
sidebar, then set ZIP_PATH. You may instead mount Google Drive yourself and set
DATA_ROOT to an existing extracted folder. Files under /content can disappear
when the runtime is reset; export results or copy them to Drive.

The optional Kaggle CLI path uses a hidden token prompt. Never paste credentials
into notebook source or share a notebook containing secrets. Downloading requires
network access, disk space, competition access and your own credentials.


In [ ]:
ZIP_PATH = ""  # Example: /content/snapshot-wi-oh-deer.zip
DOWNLOAD_WITH_KAGGLE = False
if MODE == "real":
    if DOWNLOAD_WITH_KAGGLE:
        import getpass
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
        token = getpass.getpass("Kaggle API token (hidden): ")
        env = os.environ.copy(); env["KAGGLE_API_TOKEN"] = token
        download_dir = PROJECT / "downloads"; download_dir.mkdir(exist_ok=True)
        try:
            subprocess.run(["kaggle", "competitions", "download", "-c", "snapshot-wi-oh-deer",
                            "-p", str(download_dir)], check=True, env=env)
        finally:
            del token; env.pop("KAGGLE_API_TOKEN", None)
        downloaded = sorted(download_dir.glob("*.zip"))
        if len(downloaded) != 1:
            raise RuntimeError("Set ZIP_PATH explicitly after inspecting the download directory")
        ZIP_PATH = str(downloaded[0])
    if ZIP_PATH:
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH) as archive:
            for member in archive.infolist():
                target = (DATA_ROOT / member.filename).resolve()
                if not target.is_relative_to(DATA_ROOT.resolve()):
                    raise ValueError("Unsafe dataset ZIP path")
            archive.extractall(DATA_ROOT)
    if not DATA_ROOT.exists():
        raise FileNotFoundError("Extract the real dataset and configure DATA_ROOT first")
    run("inspect", "--root", DATA_ROOT)


## Snapshot Wisconsin v2 mapping and provisional split

The downloaded v2 CSV was inspected: normalized bbox_origin_x/y and
bbox_width/height, filename, and camera_location_seq_no. The adapter locates
the Deer folder and converts these boxes to pixel xyxy automatically.
It keeps camera groups separate and uses seed 566. Supply the team split
when available; otherwise treat this split as provisional.

There is one supplied box per image, but additional unannotated deer were
observed. Do NOT claim full human ground-truth coverage. The pilot explicitly
allows background outside supplied boxes as a noisy-label assumption, disables
hard-negative mining, and records this limitation. It randomly selects up to
200 training and 100 validation images. The corrected configuration uses 64 background samples per image, up to eight scan-aligned positives per box, 256 Haar features, and an unweighted SVM from the group-separated split.
For other datasets, use the generic CSV/COCO/manifest CLI adapters in docs.


In [ ]:
SHARED_SPLIT = ""  # Optional team CSV: image_id,split
if MODE == "real":
    MANIFEST = PROJECT / "data" / "manifest.json"
    arguments = ["prepare-kaggle", "--root", DATA_ROOT, "--output", MANIFEST]
    if SHARED_SPLIT:
        arguments += ["--shared-split", SHARED_SPLIT]
    run(*arguments)
    summary = json.loads((MANIFEST.parent / "dataset_summary.json").read_text())
    IMAGE_ROOT = Path(summary["image_root"])
    print(json.dumps(summary, indent=2))
    print("WARNING: one supplied box per image; additional deer may be unannotated.")
    print("Pilot metrics measure supplied-box agreement, NOT fully annotated detection accuracy.")


## Train detect and evaluate

Training uses only train samples, including hard-negative mining and scaling.
Validation metrics are computed at IoU .5 with one-to-one matching. Default
operating threshold is SVM score >= 0. AP is single-IoU AP with the recorded
post-NMS proposal floor/cap, not COCO mAP. Inspect runtime metadata before
comparing timings from different computers.


In [ ]:
if MODE == "demo":
    run("demo", "--output", OUTPUT)
else:
    CONFIG = PROJECT / "configs" / "kaggle_corrected.json"  # Start small; use default.json for fuller training.
    run("run", "--manifest", MANIFEST, "--image-root", IMAGE_ROOT,
        "--config", CONFIG, "--output", OUTPUT)


In [ ]:
from IPython.display import display, Markdown
metrics = json.loads((OUTPUT / "metrics.json").read_text())
print(json.dumps({k:v for k,v in metrics.items() if not k.endswith("_curve")}, indent=2))
display(Markdown((OUTPUT / "report.md").read_text()))
from PIL import Image
display(Image.open(OUTPUT / "precision_recall.png"))
for example in sorted(OUTPUT.glob("example_*.jpg"))[:4]:
    image = Image.open(example); image.thumbnail((900, 650)); display(image)
if MODE == "demo":
    assert metrics["dataset_kind"] == "synthetic" and not metrics["real_data_pipeline_executed"]
    print("SOFTWARE CHECK ONLY — do not put these metrics in the midterm as real results.")


In [ ]:
import shutil
archive_path = shutil.make_archive(str(PROJECT / ("results_" + MODE)), "zip", OUTPUT)
print("Result archive:", archive_path)
DOWNLOAD_RESULTS = False  # Set True to download from a Google Colab session.
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive_path)


## What to send to teammates

After a REAL run, share the split CSV, configuration, metrics.json,
predictions.csv, inference.json and representative examples. Fill in the
baseline report template using real observations. Keep dataset images,
models, credentials and local environments out of GitHub. Do not present
synthetic checks or locally executed Notebook tests as cloud/real-data results.
